# 7B QLoRA Fine-tune on SQL Training Set

This kernel fine-tunes **Qwen/Qwen2.5-7B-Instruct** on your **SQL training set** (2500 examples)
with [QLoRA](https://arxiv.org/abs/2305.14314) (4-bit base weights + LoRA adapters), so it fits
on a single Kaggle T4/P100 16 GB GPU.

**Pipeline:** locate project files -> install deps -> resolve dataset -> prepare chat JSONL ->
train -> save adapter + metrics -> optional Hugging Face push.

**Before running:**
1. Set this kernel to **GPU** (Runtime -> Change runtime type -> GPU).
2. Attach your SQL training set as a Kaggle dataset (or push with `./scripts/push_kernel.sh`).
3. Optional: add `HF_TOKEN` as a Kaggle Secret to push the adapter after training.

In [ ]:
# 1) Locate the project files bundled with this kernel (src/, config.yaml, data/)
import glob, os

candidates = (
    glob.glob('/kaggle/input/*/src/train.py')
    + glob.glob('/kaggle/input/*/*/src/train.py')
    + glob.glob('/kaggle/working/src/train.py')
    + glob.glob('src/train.py')
)
if not candidates:
    raise FileNotFoundError(
        'project files not found - push with scripts/push_kernel.sh, or attach the '
        'project folder as kernel data'
    )
REPO = os.path.dirname(os.path.dirname(os.path.abspath(candidates[0])))
with open('/kaggle/working/.repo', 'w') as fh:
    fh.write(REPO)
print('project root:', REPO)
print(sorted(os.listdir(REPO)))

In [ ]:
# 2) Install dependencies (preinstalled Kaggle packages are reused)
import os, subprocess, sys

REPO = open('/kaggle/working/.repo').read().strip()
req = os.path.join(REPO, 'requirements.txt')
if os.path.exists(req):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '-r', req]
else:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q',
           'transformers>=4.46', 'datasets', 'accelerate', 'peft', 'trl',
           'bitsandbytes', 'pyyaml', 'python-dotenv']
subprocess.run(cmd, check=True)

import torch, transformers, peft, trl
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
print('transformers', transformers.__version__, '| peft', peft.__version__, '| trl', trl.__version__)
assert torch.cuda.is_available(), 'Enable GPU: Runtime -> Change runtime type -> GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 3) Optional secrets + resolve the SQL training set
#    precedence: TRAIN_DATASET_PATH secret/env > dataset attached to kernel > bundled sample data
import glob, os

REPO = open('/kaggle/working/.repo').read().strip()

try:
    from kaggle_secrets import UserSecretsClient
    for key in ('HF_TOKEN', 'HF_MODEL_REPO', 'WANDB_API_KEY'):
        try:
            val = UserSecretsClient().get_secret(key)
            if val:
                os.environ[key] = val
                print(f'secret {key}: loaded')
        except Exception:
            pass
except Exception as ex:
    print('kaggle_secrets unavailable:', ex)

data_path = os.environ.get('TRAIN_DATASET_PATH')
if not data_path:
    # Look for JSON files recursively in /kaggle/input (handles subdirs like part1/, part2/)
    attached = [p for p in (glob.glob('/kaggle/input/**/*.json', recursive=True)
                            + glob.glob('/kaggle/input/**/*.jsonl', recursive=True)
                            + glob.glob('/kaggle/input/**/*.csv', recursive=True))
                if not p.startswith(REPO)]
    if attached:
        # Use the parent directory of the first data file found
        data_path = os.path.dirname(attached[0])
        # If files are in subdirs (part1/, part2/), go up to the common root
        while data_path != '/kaggle/input' and not os.path.basename(data_path).startswith('part'):
            parent = os.path.dirname(data_path)
            if parent == '/kaggle/input':
                break
            data_path = parent
    else:
        data_path = os.path.join(REPO, 'data', 'raw')
with open('/kaggle/working/.datapath', 'w') as fh:
    fh.write(data_path)
print('training data:', data_path)
if os.path.isdir(data_path):
    all_files = sorted(glob.glob(os.path.join(data_path, '**', '*.json'), recursive=True))
    print(f'found {len(all_files)} JSON files')
    for f in all_files[:5]:
        print(f'  - {os.path.relpath(f, data_path)}')
    if len(all_files) > 5:
        print(f'  ... and {len(all_files) - 5} more')
else:
    print('data file:', data_path)

In [ ]:
# 4) Validate dataset format and count records
import glob, json, os

DATA = open('/kaggle/working/.datapath').read().strip()

if os.path.isdir(DATA):
    files = sorted(glob.glob(os.path.join(DATA, '**', '*.json'), recursive=True))
else:
    files = [DATA]

total_records = 0
format_ok = True
for f in files:
    try:
        data = json.load(open(f, encoding='utf-8'))
        if isinstance(data, list):
            total_records += len(data)
            if data and not all('instruction' in r and 'output' in r for r in data[:3]):
                format_ok = False
                print(f'  [warn] {os.path.basename(f)}: missing instruction/output keys')
        elif isinstance(data, dict):
            total_records += 1
    except Exception as e:
        print(f'  [error] {os.path.basename(f)}: {e}')
        format_ok = False

print(f'\nTotal records: {total_records}')
print(f'Format (Alpaca instruction/output): {"OK" if format_ok else "CHECK WARNINGS ABOVE"}')
assert total_records > 0, 'No records found!'
print('Dataset ready for preparation.')

In [ ]:
# 5) Prepare data -> chat-format train/val JSONL
import os, subprocess, sys

REPO = open('/kaggle/working/.repo').read().strip()
DATA = open('/kaggle/working/.datapath').read().strip()
cmd = [sys.executable, os.path.join(REPO, 'src', 'prepare_data.py'),
       '--input', DATA, '--output', os.path.join(REPO, 'data', 'processed'),
       '--val-split', '0.05']
subprocess.run(cmd, check=True)

In [ ]:
# 6) QLoRA fine-tuning (set a MAX_STEPS secret/env var for a quick smoke test)
import os, subprocess, sys

REPO = open('/kaggle/working/.repo').read().strip()
OUTPUT = '/kaggle/working/outputs'
cmd = [sys.executable, os.path.join(REPO, 'src', 'train.py'),
       '--config', os.path.join(REPO, 'config.yaml'), '--output-dir', OUTPUT]
if os.environ.get('MAX_STEPS'):
    cmd += ['--max-steps', os.environ['MAX_STEPS']]
print('running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 7) Inspect results (adapter, checkpoints, metrics)
import glob, json, os

paths = sorted(glob.glob('/kaggle/working/outputs/**', recursive=True))
for p in paths[:60]:
    print(p)
print(f'... {len(paths)} paths total')
for name in ('train_results.json', 'eval_results.json', 'run_config.json'):
    path = os.path.join('/kaggle/working/outputs', name)
    if os.path.exists(path):
        print(f'\n== {name} ==')
        print(json.dumps(json.load(open(path)), indent=2, default=str))

In [ ]:
# 8) Optional: push the trained adapter to Hugging Face
#    (add HF_TOKEN and HF_MODEL_REPO as Kaggle Secrets to enable)
import os

repo = os.environ.get('HF_MODEL_REPO', '')
token = os.environ.get('HF_TOKEN', '')
adapter = '/kaggle/working/outputs/adapter'
if repo and token and 'your_hf_username' not in repo and os.path.isdir(adapter):
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(repo_id=repo, exist_ok=True)
    api.upload_folder(repo_id=repo, folder_path=adapter)
    print('pushed adapter to https://huggingface.co/' + repo)
else:
    print('HF push skipped - set HF_TOKEN + HF_MODEL_REPO secrets to enable')

## Next steps

- **Download artifacts:** this notebook's output (`outputs/adapter/`) is available via
  *Output -> Download* on Kaggle, or run `./scripts/pull_results.sh` locally.
- **Chat with the model locally:**
  ```bash
  python src/inference.py --interactive --adapter outputs/adapter
  ```
- **Tune hyperparameters:** edit `config.yaml` (LoRA rank, LR, epochs, seq length) and re-push.